# Distribution Feeder Modeling

This section describes distribution feeder modeling concepts and the CIMantic Graphs `FeederModel` class for accessing CIM feeder models.

All diagrams on this page are generated as [Mermaid](https://mermaid.js.org/) text with the `cimgraph.utils` module of CIMantic Graphs and displayed as a Markdown `mermaid` block:

In [1]:
from IPython.display import Markdown, display

from cimgraph import utils

def show_mermaid(diagram):
    display(Markdown(f'```mermaid\n{diagram}\n```'))

## Distribution Feeders in CIM

Full modeling of North American distribution circuits was introduced in CIM 17 to include unbalanced medium-voltage networks, low-voltage customer-side equipment, DERs, and advanced metering infrastructure (AMI) usage points. An example of a very simple distribution feeder is shown and explained below:

![basic-feeder](./images/4_2_feeder.png)

* `ConnectivityNode` used for 3-phase buses. Separate modeling of single-phase nodes is not used

* `Terminal` used to connect conducting equipment to a bus

* `ACLineSegment` used for overhead lines, underground cables, and secondary triplex lines. Phasing represented using `ACLineSegmentPhase`

* Downstream switching devices represented using `Recloser`, `Sectionaliser`, etc.

* Poletop `LinearShuntCompensator` and `RatioTapChanger` devices

* `EnergySource` slack bus at substation

* Loads can be represented at medium voltage level or broken down into individual `EnergyConsumer` customers with split-phase `TransformerTank`

----

## FeederModel Class

The `FeederModel` class in CIMantic Graphs provides the interface for users to access a single feeder at a time (in a manner similar to study tools such as EPRI OpenDSS or Eaton CYME). The class provides access to the typed property graph and API methods to read and write to a data source.

It can be imported using

In [2]:
from cimgraph.models import FeederModel

The structure of the distribution network graphs and API methods are explained in the sections below.

----

## Loading a FeederModel from an XML File

When working with small test cases (e.g. IEEE 13 and 123 bus models), CIMantic Graphs can be used without any database by building the graph directly from an XML file. *Note: large models (e.g. IEEE 8500) take longer to read from XML, as every triple is loaded and the full network graph is built at once.*

The first step is to import the CIM profile and set the matching environment variable used by CIM-Graph. For an explanation of CIM profiles, see [Profiles Overview](../02_cim_profiles/2_1_profiles_overview.ipynb).

In [3]:
import os
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
import cimprofiles.cim17.cimhub_2023 as cim

Next, use the `XMLFile` interface to open the XML file:

In [4]:
from cimgraph.databases import XMLFile
file = XMLFile(filename='../../tests/test_models/ieee13.xml')

Finally, create a new `FeederModel` with the feeder's mRID as the container to load the feeder into the graph:

In [5]:
feeder = cim.Feeder(mRID='49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
network = FeederModel(container=feeder, connection=file)

The complete network model, including both forward and reverse associations between all objects, has now been loaded into the FeederModel graph. For example, the loads and their power in kW (unit values are converted with `.to()`):

In [6]:
for load in network.list_by_class(cim.EnergyConsumer)[:5]:
    print(load.name, load.p.to('kW'), 'kW', load.q.to('kVAr'), 'kVAr')

646 230.0 kW 132.0 kVAr
house 5.0 kW 3.0987217 kVAr
634c 120.0 kW 90.0 kVAr
675c 290.0 kW 212.0 kVAr
645 170.0 kW 125.0 kVAr


To show the structure of the resulting graph, we can display a sample breaker object:

In [7]:
# Get first breaker in graph
breaker = network.first(cim.Breaker)
# Display the breaker
show_mermaid(utils.get_mermaid(breaker))

```mermaid
mindmap
    52DE9189((**Breaker**
        name: brkr1
        normalOpen: False
        open: False
        retained: True
        ratedCurrent: 400 ampere
        breakingCapacity: 400 ampere))
        [Location]
            085BBE1F((**Location**
                name: brkr1_Loc))
        [EquipmentContainer]
            49AD8E07((**Feeder**
                name: ieee13nodeckt))
        [BaseVoltage]
            2A158E0C((**BaseVoltage**
                name: BaseV_4.1600))
        ["Terminals"]
            1D81C7FE((**Terminal**
                name: brkr1_T1))
            2847E06B((**Terminal**
                name: brkr1_T2))

```

----

## Loading a FeederModel from a Database

When working with large models (e.g. IEEE 8500/9500 node or utility networks), it is generally advisable to load the models into a database and then use CIM-Graph to access the feeder via its API. The example below needs a running Blazegraph database with the IEEE 13 model loaded and the `sparql` extra (`pip install cim-graph[sparql]`), so it is not run here. See [Databases Overview](../03_databases/3_1_databases_overview.ipynb) for the other connections.

As before, import the CIM profile and set the environment variable. Then connect to the database:

```python
import os
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
os.environ['CIMG_URL'] = 'http://localhost:8889/bigdata/namespace/kb/sparql'
import cimprofiles.cim17.cimhub_2023 as cim
from cimgraph.databases import BlazegraphConnection

database = BlazegraphConnection()
```

Next, create a CIM `Feeder` object with the model mRID, or retrieve it from the database with `.get_object()`:

```python
feeder = cim.Feeder(mRID='49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
# or
feeder = database.get_object(mRID='49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
```

Finally, create a new `FeederModel` and pass in the feeder and database. This loads the base feeder topology into the graph:

```python
network = FeederModel(container=feeder, connection=database)
```

To ensure performance on large models, only the minimum set of `Equipment`, `ConnectivityNode`, and `Terminal` objects are loaded into the graph, with only their identifiers. All other attributes are left as `None` or `[]` until the object classes are populated using the `.get_all_edges()` method (see Graph Expansion below).

----

## FeederModel.graph

The `.graph` attribute of the GraphModel dataclass is the typed property graph with all data contained in the network model.

It is a dictionary keyed first by class (e.g. `cim.ACLineSegment`) and then by the UUID identifier of each object.

All objects within the graph (if added using the API) are forced to have a unique identifier.

### Graph Initialization

The CIM `Feeder` class describes the containment and naming of equipment and nodes within distribution feeders. For most applications, each feeder has a single normal energizing substation that serves as the source for all downstream customers. A database connection initializes the graph with the classes below:

In [8]:
diagram_text = utils.get_mermaid_path(cim.Feeder, "Equipments", show_attributes=False)
diagram_text = utils.add_mermaid_path(cim.Feeder, "ConnectivityNodes", diagram_text, show_attributes=False)
diagram_text = utils.add_mermaid_path(cim.Feeder, "NormalEnergizingSubstation", diagram_text, show_attributes=False)
diagram_text = utils.add_mermaid_path(cim.Equipment, "Measurements", diagram_text, show_attributes=False)
diagram_text = utils.add_mermaid_path(cim.ConnectivityNode, "Terminals", diagram_text, show_attributes=False)

show_mermaid(diagram_text)

```mermaid
%%{init: {"theme":"neutral"}}%%
classDiagram
    class Feeder{
    }
    Feeder --o "0..*" Equipment : Equipments 
    class Equipment{
    }
    Feeder --> "0..*" ConnectivityNode : ConnectivityNodes 
    class ConnectivityNode{
    }
    Feeder --> "0..1" Substation : NormalEnergizingSubstation 
    class Substation{
    }
    Equipment --o "0..*" Measurement : Measurements 
    class Measurement{
    }
    ConnectivityNode --> "0..*" Terminal : Terminals 
    class Terminal{
    }

```

----

### Graph Expansion

One of the key features of CIMantic Graphs is the elimination of custom database queries to access information about equipment, impedances, measurements etc. Instead, a single API method extracts all information about a class.

The `.get_all_edges()` method is the core library method that enables CIM-Graph to query for CIM objects of any class and build the knowledge graph without custom queries.

The arguments of the method are

* `cim_class` (type): The CIM class for which to retrieve edges (e.g. `cim.Breaker`)

With a database connection, the breaker above starts with only its identifier. After the call below, it has all its attributes and associations:

```python
network.get_all_edges(cim.Breaker)
show_mermaid(utils.get_mermaid(breaker))
```

The file parsers (`XMLFile`, `JSONLDFile`) read the whole file when the model is created, so for them `.get_all_edges()` does nothing.

----

## Distributed Feeder Models

`FeederModel(..., distributed=True)` splits the feeder into a hierarchy of feeder, switch and secondary areas. See [Distributed Area](4_5_distributed_area.ipynb).